# Using OpenAI as Your Model Provider

OpenAI is the default model provider in MemoRizz. This notebook shows how to configure
and use OpenAI models (GPT-4o, GPT-4o Mini, etc.) with your MemoRizz agents.

**What you'll learn:**
1. Setting up the OpenAI provider via config dict or direct instantiation
2. Configuring model parameters (temperature, max_tokens, etc.)
3. Using different OpenAI models for different use cases
4. Streaming responses

> **Prerequisites:** An OpenAI API key. Get one at [platform.openai.com](https://platform.openai.com)

In [ ]:
%pip install -qU memorizz openai

In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("API key configured.")

---
## Method 1: Config Dict (Recommended)

The simplest way — pass a config dict to `MemAgentBuilder.with_llm_config()`. The factory
creates the provider automatically.

In [ ]:
from memorizz.memagent.builders import MemAgentBuilder

agent = (
    MemAgentBuilder()
    .with_instruction("You are a helpful assistant.")
    .with_llm_config({
        "provider": "openai",
        "model": "gpt-4o-mini",
        "temperature": 0.7,
    })
    .build()
)

response = agent.run("What are the three laws of robotics?")
print(response)

## Method 2: Direct Provider Instance

For more control, instantiate the provider directly and pass it with `.with_model()`.

In [ ]:
from memorizz.llms import OpenAI
from memorizz.memagent.builders import MemAgentBuilder

llm = OpenAI(
    model="gpt-4o",
    temperature=0.3,
    max_tokens=2000,
    top_p=0.95,
)

agent = (
    MemAgentBuilder()
    .with_instruction("You are a concise technical writer.")
    .with_model(llm)
    .build()
)

response = agent.run("Explain the CAP theorem in 3 sentences.")
print(response)

## Streaming Responses

OpenAI supports streaming out of the box. Use `agent.run_stream()` to get tokens as they arrive.

In [ ]:
for event in agent.run_stream("Write a haiku about Python programming."):
    if event.get("type") == "content":
        print(event["content"], end="", flush=True)
print()

## Config Reference

| Key | Type | Default | Description |
|-----|------|---------|-------------|
| `provider` | str | `"openai"` | Must be `"openai"` |
| `model` | str | `"gpt-4o"` | Model ID (e.g. `gpt-4o`, `gpt-4o-mini`, `gpt-4-turbo`) |
| `api_key` | str | `OPENAI_API_KEY` env var | API key (optional if env var is set) |
| `temperature` | float | None | Sampling temperature (0.0-2.0) |
| `max_tokens` | int | None | Maximum tokens to generate |
| `top_p` | float | None | Nucleus sampling |
| `frequency_penalty` | float | None | Frequency penalty (-2.0 to 2.0) |
| `presence_penalty` | float | None | Presence penalty (-2.0 to 2.0) |
| `seed` | int | None | Random seed for reproducibility |

## Token Usage

After each run, you can inspect token usage and context-window stats.

In [ ]:
stats = agent.get_context_window_stats()
print(f"Context window: {stats}")